In [1]:
from pathlib import Path
import pandas as pd
from joblib import Parallel, delayed
from collections import defaultdict
import pybedtools

In [2]:
ROOT = Path().cwd()
DATA_INITIAL = Path("../data_initial").resolve()
DATA = Path("../data").resolve()
RNACHROM_INITIAL = DATA_INITIAL / "rnachrom"
PREPROCESSING = Path("../preprocessing")
RNACHROM_INDEX_BED = PREPROCESSING / "rnachrom_index-bed"

In [3]:
for directory in DATA_INITIAL, DATA, RNACHROM_INITIAL, PREPROCESSING, RNACHROM_INDEX_BED:
    directory.mkdir(exist_ok=True, parents=True)

In [4]:
to_himorna_rnachrom_mapping = RNACHROM_INITIAL / "lncRNA_CorrespondenceTable.tsv"
himorna_rnachrom_mapping = pd.read_csv(to_himorna_rnachrom_mapping, sep="\t")
himorna_rnachrom_mapping.head(2)

,chr_rna HimoRNA,start_rna HimoRNA,end_rna HimoRNA,str,gene_name HimoRNA,Ensembl ID HimoRNA,chr_rna RNA-Chrom,start_rna RNA-Chrom,end_rna RNA-Chrom,gene_name RNA-Chrom,Ensembl ID RNA-Chrom,Jaccard index
0,chr1,89295,133723,-,AL627309.1,ENSG00000238009.6,chr1,89295.0,133723.0,AL627309.1,ENSG00000238009.6,1.0
1,chr1,89551,91105,-,AL627309.3,ENSG00000239945.1,chr1,89551.0,91105.0,AL627309.3,ENSG00000239945.1,1.0


In [5]:
himorna_rnachrom_mapping.shape

(4149, 12)

In [6]:
himorna_rnachrom_mapping = himorna_rnachrom_mapping[["gene_name HimoRNA", "Ensembl ID HimoRNA", "gene_name RNA-Chrom", "Ensembl ID RNA-Chrom"]]

In [7]:
%%time
def drop_ensembl_version(x):
    return x.split(".")[0] if isinstance(x, str) else x

for col in ["Ensembl ID HimoRNA", "Ensembl ID RNA-Chrom"]:
    himorna_rnachrom_mapping[col] = himorna_rnachrom_mapping[col].apply(drop_ensembl_version)

CPU times: user 9.05 ms, sys: 1 ms, total: 10 ms
Wall time: 9.46 ms


In [8]:
himorna_rnachrom_mapping

,gene_name HimoRNA,Ensembl ID HimoRNA,gene_name RNA-Chrom,Ensembl ID RNA-Chrom
0,AL627309.1,ENSG00000238009,AL627309.1,ENSG00000238009
1,AL627309.3,ENSG00000239945,AL627309.3,ENSG00000239945
2,AL627309.2,ENSG00000239906,AL627309.2,ENSG00000239906
3,AL669831.5,ENSG00000237491,LINC01409,ENSG00000237491
4,FAM87B,ENSG00000177757,FAM87B,ENSG00000177757
...,...,...,...,...
4144,AL080317.1,ENSG00000230177,NaN,NaN
4145,AL356599.1,ENSG00000235652,NaN,NaN
4146,NKAIN3-IT1,ENSG00000274956,NaN,NaN
4147,AC010980.1,ENSG00000267034,NaN,NaN


In [9]:
himorna_rnachrom_mapping["Ensembl ID HimoRNA"].nunique()

4145

In [10]:
himorna_rnachrom_mapping["Ensembl ID RNA-Chrom"].nunique()

4120

In [11]:
contacts_test = pd.concat([pd.read_csv(i, sep="\t", usecols=["dna_chr","dna_start","dna_end","gene_name","gene_strand"]) for i in RNACHROM_INITIAL.glob("experiment_all_to_all_id_1_method_iMARGI_organism_Homo_sapiens_tissue_HFFc6/contacts/*.tab")])
contacts_test.head(2)

,dna_chr,dna_start,dna_end,gene_name,gene_strand
0,chr3,47595961,47596017,SMARCC1,-
1,chr7,139874781,139874838,ATP13A3,-


In [12]:
contacts_test

,dna_chr,dna_start,dna_end,gene_name,gene_strand
0,chr3,47595961,47596017,SMARCC1,-
1,chr7,139874781,139874838,ATP13A3,-
2,chr3,47851695,47851752,MAP4,-
3,chr3,72631180,72631234,RYBP,-
4,chr1,243368102,243368155,ATG7,+
...,...,...,...,...,...
64047,chrX,124054111,124054210,STAG2,+
64048,chrX,129945039,129945138,AL034405.1,-
64049,chr5,5321181,5321246,RPS6KA6,-
64050,chr11,13268429,13268472,NONO,+


In [13]:
dd = pd.merge(left=contacts_test, right=himorna_rnachrom_mapping, how="inner", left_on="gene_name", right_on="gene_name RNA-Chrom")
dd = dd[["dna_chr","dna_start","dna_end","gene_strand","Ensembl ID HimoRNA"]]
dd

,dna_chr,dna_start,dna_end,gene_strand,Ensembl ID HimoRNA
0,chr3,47215955,47216025,+,ENSG00000227398
1,chr5,154224456,154224492,+,ENSG00000240057
2,chr3,40265787,40265838,-,ENSG00000280739
3,chr3,181253813,181253849,+,ENSG00000242808
4,chr3,178303536,178303636,-,ENSG00000223930
...,...,...,...,...,...
177404,chrX,20207452,20207501,-,ENSG00000230590
177405,chr5,87806274,87806341,-,ENSG00000230590
177406,chrX,63480664,63480764,-,ENSG00000235437
177407,chrX,115917470,115917570,-,ENSG00000235244


In [14]:
dd["Ensembl ID HimoRNA"].nunique()

3182

In [105]:
dict_of_rnachrom_peaks_by_exp_tissue = {}
for project in RNACHROM_INITIAL.glob("experiment*"):
    if project.is_dir:
        print(project.name)
        exp = project.name.split("method_")[1].split(("_organism"))[0]
        print(exp)
        tissue = project.name.split("tissue_")[1]
        print(tissue)
        res_concat = pd.concat([pd.read_csv(i, sep="\t", usecols=["dna_chr","dna_start","dna_end","gene_name","gene_strand"]) for i in RNACHROM_INITIAL.glob(f"{project.name}/contacts/*.tab")])
        dd = pd.merge(left=res_concat, right=himorna_rnachrom_mapping, how="inner", left_on="gene_name", right_on="gene_name RNA-Chrom")
        dict_of_rnachrom_peaks_by_exp_tissue[f"{exp}-{tissue}"] = dd

experiment_all_to_all_id_1_method_iMARGI_organism_Homo_sapiens_tissue_HFFc6
iMARGI
HFFc6
experiment_all_to_all_id_8_method_GRID_organism_Homo_sapiens_tissue_MDA_MB_231__HTB_26_ATCC_
GRID
MDA_MB_231__HTB_26_ATCC_
experiment_all_to_all_id_9_method_Red_C_organism_Homo_sapiens_tissue_fibroblasts
Red_C
fibroblasts
experiment_all_to_all_id_2_method_iMARGI_organism_Homo_sapiens_tissue_HEK293T
iMARGI
HEK293T
experiment_all_to_all_id_3_method_iMARGI_organism_Homo_sapiens_tissue_HUVEC
iMARGI
HUVEC
experiment_all_to_all_id_4_method_iMARGI_organism_Homo_sapiens_tissue_HUVEC
iMARGI
HUVEC
experiment_all_to_all_id_5_method_iMARGI_organism_Homo_sapiens_tissue_HUVEC
iMARGI
HUVEC
experiment_all_to_all_id_7_method_GRID_organism_Homo_sapiens_tissue_MM_1S
GRID
MM_1S
experiment_all_to_all_id_10_method_Red_C_organism_Homo_sapiens_tissue_K562
Red_C
K562


In [106]:
dict_of_rnachrom_peaks_by_exp_tissue.keys()

dict_keys(['iMARGI-HFFc6', 'GRID-MDA_MB_231__HTB_26_ATCC_', 'Red_C-fibroblasts', 'iMARGI-HEK293T', 'iMARGI-HUVEC', 'GRID-MM_1S', 'Red_C-K562'])

In [107]:
def indexing_rnachrom_lncrna_by_bed(exp_tissue, lnc_ens, subdf):
    to_index = RNACHROM_INDEX_BED / exp_tissue
    to_index.mkdir(exist_ok=True, parents=True)
    to_save = to_index / f"{exp_tissue}-{lnc_ens}-sort.bed"
    if to_save.exists():
        print(f"Skipped: {exp_tissue} {lnc_ens}")
        return None
    subdf["name"] = "."
    subdf["score"] = "."
    subdf = subdf[["dna_chr", "dna_start", "dna_end", "name", "score", "gene_strand"]]
    subdf_bed = pybedtools.BedTool.from_dataframe(subdf)
    subdf_bed_sortbed = subdf_bed.sort()
    subdf_bed_sortbed.moveto(to_save)

In [108]:
with Parallel(n_jobs=-1) as pool:
    chunks = pool(
        delayed(indexing_rnachrom_lncrna_by_bed)(exp_tissue=exp_tissue, lnc_ens=lnc_ens[0], subdf=subdf)
        for exp_tissue, dd in dict_of_rnachrom_peaks_by_exp_tissue.items()
        for lnc_ens, subdf in dd.groupby(by=["Ensembl ID HimoRNA"])
    )

In [88]:
for lnc_ens, subdf in dd.groupby(by=["Ensembl ID HimoRNA"]):
    print(lnc_ens)
    break

('ENSG00000093100',)


In [113]:
[pd.read_csv(i) for i in (RNACHROM_INDEX_BED / "GRID-MDA_MB_231__HTB_26_ATCC_").glob("GRID-MDA_MB_231__HTB_26_ATCC_-ENSG00000093100-sort.bed")]

[          chr1\t8045642\t8045665\t.\t.\t-
 0         chr1\t9584333\t9584355\t.\t.\t-
 1         chr1\t9720817\t9720840\t.\t.\t-
 2       chr1\t10230041\t10230063\t.\t.\t-
 3       chr1\t17878112\t17878134\t.\t.\t-
 4       chr1\t17878112\t17878134\t.\t.\t-
 ...                                   ...
 1369  chrX\t135004207\t135004230\t.\t.\t-
 1370  chrX\t135179552\t135179574\t.\t.\t-
 1371  chrX\t136145565\t136145587\t.\t.\t-
 1372  chrX\t146357759\t146357781\t.\t.\t-
 1373  chrX\t152810062\t152810085\t.\t.\t-
 
 [1374 rows x 1 columns]]

## Аннотация (вроде бы излишняя)

In [10]:
gtf_file_GENCODE35 = DATA_INITIAL / "gencode/gencode.v35.basic.annotation.gtf.gz"
anno_GENCODE35 = pd.read_csv(gtf_file_GENCODE35, sep="\t", header=None, skiprows=5)
anno_GENCODE35.head(2)

,0,1,2,3,4,5,6,7,8
0,chr1,HAVANA,gene,11869,14409,.,+,.,"gene_id ""ENSG00000223972.5""; gene_type ""transc..."
1,chr1,HAVANA,transcript,11869,14409,.,+,.,"gene_id ""ENSG00000223972.5""; transcript_id ""EN..."


In [11]:
anno_GENCODE35.shape

(1789637, 9)

In [12]:
anno_GENCODE35["ens_gene"] = anno_GENCODE35[8].apply(lambda x: x.split(" ")[1].replace('"', '').replace(';', '').split(".")[0])

In [13]:
anno_GENCODE35["gene_name"] = anno_GENCODE35[8].apply(lambda x: x.split("gene_name ")[1].split(";")[0].replace('"', ''))

In [14]:
ens_and_name_GENCODE35 = anno_GENCODE35[["ens_gene", "gene_name"]].drop_duplicates()
#ens_and_name_GENCODE35.set_index("gene_name", inplace=True)
ens_and_name_GENCODE35.head(2)

,ens_gene,gene_name
0,ENSG00000223972,DDX11L1
12,ENSG00000227232,WASH7P


In [109]:
ens_and_name_GENCODE35.shape

(60612, 2)

In [15]:
len([i for i in RNACHROM_INITIAL.glob("*/contacts/*.tab")])

363

In [16]:
pd.read_csv([i for i in RNACHROM_INITIAL.glob("*/contacts/*.tab")][0], sep="\t")

,rna_chr,rna_start,rna_end,dna_chr,dna_start,dna_end,sample,gene_name,gene_type,gene_strand,source,expected_value,norm_value
0,chr3,47595901,47595999,chr3,47595961,47596017,SRR8206680,SMARCC1,protein_coding,-,gencode,0.03999,1.007
1,chr3,194463805,194463903,chr7,139874781,139874838,SRR8206680,ATP13A3,protein_coding,-,gencode,0.06362,0.964
2,chr3,47851633,47851731,chr3,47851695,47851752,SRR8206680,MAP4,protein_coding,-,gencode,0.02933,1.027
3,chr3,72421268,72421366,chr3,72631180,72631234,SRR8206680,RYBP,protein_coding,-,gencode,0.02967,1.026
4,chr3,11315978,11316029,chr1,243368102,243368155,SRR8206680,ATG7,protein_coding,+,gencode,0.02830,1.029
...,...,...,...,...,...,...,...,...,...,...,...,...,...
242889,chr3,184819152,184819181,chr3,135445357,135445455,SRR8206680,VPS8,protein_coding,+,gencode,0.00143,1.084
242890,chr3,37045936,37046034,chr3,37045871,37045971,SRR8206680,MLH1,protein_coding,+,gencode,0.10700,0.895
242891,chr3,86005261,86005345,chr3,86005376,86005437,SRR8206680,CADM2-AS1,lncRNA,-,gencode,0.00102,1.085
242892,chr3,53868490,53868588,chr16,59640738,59640780,SRR8206680,ACTR8,protein_coding,-,gencode,0.02338,1.038


In [17]:
test1_chr3 = pd.read_csv([i for i in RNACHROM_INITIAL.glob("experiment_all_to_all_id_1_method_iMARGI_organism_Homo_sapiens_tissue_HFFc6/contacts/*.tab")][0], sep="\t")
test1_chr3.shape

(242894, 13)

In [18]:
ens_and_name_GENCODE35[ens_and_name_GENCODE35["gene_name"] == "SMARCC1"]

,ens_gene,gene_name
332370,ENSG00000173473,SMARCC1


In [115]:
merge = pd.merge(left=ens_and_name_GENCODE35, right=test1_chr3, left_on="gene_name", right_on="gene_name", how="inner")

In [118]:
merge

,ens_gene,gene_name,rna_chr,rna_start,rna_end,dna_chr,dna_start,dna_end,sample,gene_type,gene_strand,source,expected_value,norm_value
0,ENSG00000223587,LINC01986,chr3,13571,13669,chr3,13485,13585,SRR8206680,lncRNA,+,gencode,0.005670,1.075
1,ENSG00000223587,LINC01986,chr3,12701,12799,chr3,12670,12770,SRR8206680,lncRNA,+,gencode,0.004240,1.078
2,ENSG00000223587,LINC01986,chr3,22374,22472,chr3,22373,22473,SRR8206680,lncRNA,+,gencode,0.038730,1.009
3,ENSG00000223587,LINC01986,chr3,22156,22254,chr3,22008,22108,SRR8206680,lncRNA,+,gencode,0.038730,1.009
4,ENSG00000223587,LINC01986,chr3,19365,19463,chr3,19247,19347,SRR8206680,lncRNA,+,gencode,0.032600,1.021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
224358,ENSG00000232783,FRG2FP,chr3,198111098,198111196,chr3,198111022,198111122,SRR8206680,transcribed_unprocessed_pseudogene,+,gencode,0.013090,1.059
224359,ENSG00000232783,FRG2FP,chr3,198111072,198111170,chr3,198111022,198111122,SRR8206680,transcribed_unprocessed_pseudogene,+,gencode,0.013090,1.059
224360,ENSG00000232783,FRG2FP,chr3,198111098,198111196,chr3,198111022,198111090,SRR8206680,transcribed_unprocessed_pseudogene,+,gencode,0.013090,1.059
224361,ENSG00000236246,AC073135.1,chr3,198121773,198121871,chr9,138178354,138178454,SRR8206680,unprocessed_pseudogene,-,gencode,0.012650,1.060


## Продолжение